In [ ]:
!pip install wfdb torchmetrics

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
from sklearn.preprocessing import MultiLabelBinarizer
import torchmetrics
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
if device.type == 'cuda':
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if device.type == 'cuda':
    !time unzip -q "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip" -d "/content/"
    !ls -lah /content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 | head

In [ ]:
if device.type == 'cuda':
    best_model_save_dir = '/content/drive/MyDrive/'
else:
    best_model_save_dir = 'models'

In [ ]:
def load_raw_data(df, sampling_rate):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

if device.type == 'cuda':
    parent_path = '/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
else:
    parent_path = os.curdir
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(os.path.join(parent_path,'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(os.path.join(parent_path,'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [107]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

In [108]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

In [109]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

((19601, 1000, 12), dtype('float64'), (19601, 5), dtype('int64'))

In [110]:
# just examining the data some
summed = np.sum(y_train, axis=1)
record_ids_with_no_labels_true = []
for id, class_sum in enumerate(summed):
    if class_sum == 0:
        record_ids_with_no_labels_true.append(id)
print(f'{len(record_ids_with_no_labels_true)} train examples have all labels as "0"')

371 train examples have all labels as "0"


In [112]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).transpose(1,2)  # the 12 channels dimension is initially loaded in as last
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32).transpose(1,2)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [113]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [114]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(12, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)

        # average the whole time series into 1 length to account for any time series length
        # while this discards temporal detail, the abnormalities are more concerned over the presence/strength of extracted features (representing the presence of abnormalities)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [115]:
# TODO: try adding pos_weight with pos_weight_c = N_neg_c / N_pos_c (where c is each of the 5 labels)
loss_fn = nn.BCEWithLogitsLoss()  

In [ ]:
def train_model(model, optimizer, loss_fn, train_loader, test_loader, best_model_save_path, num_epochs=10):
    best_loss = float('inf')  # start with infinity
    for epoch in range(num_epochs):
        running_train_loss = 0.0
        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            preds = model(x)

            loss = loss_fn(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * x.size(0)
        epoch_train_loss = running_train_loss / len(train_loader.dataset)

        running_test_loss = 0.0
        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)
                preds = model(x)
                loss = loss_fn(preds, y)
                running_test_loss += loss.item() * x.size(0)
            epoch_test_loss = running_test_loss / len(test_loader.dataset)

        print(f"Epoch {epoch+1}, Training Loss: {epoch_train_loss:.4f}, Testing Loss: {epoch_test_loss:.4f}")

        if epoch_test_loss < best_loss:
            best_loss = epoch_test_loss
            torch.save(model.state_dict(), best_model_save_path)

In [120]:
def run_inference(model, dataloader):
    model.eval()
    all_test_preds, all_test_labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_test_preds.append(preds.cpu())
            all_test_labels.append(y.cpu())

    # Concatenate all batches
    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    return all_test_preds, all_test_labels

In [ ]:
model = CNN(num_classes=y_train_tensor.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, 'cnn_best_model.pt'), num_epochs=30)

Epoch 1, Training Loss: 0.3620, Testing Loss: 0.33756485039044554
Epoch 2, Training Loss: 0.3088, Testing Loss: 0.33028120610581624
Epoch 3, Training Loss: 0.2933, Testing Loss: 0.3234393212662923
Epoch 4, Training Loss: 0.2864, Testing Loss: 0.31697529915032113
Epoch 5, Training Loss: 0.2786, Testing Loss: 0.3075056166351655
Epoch 6, Training Loss: 0.2742, Testing Loss: 0.30446352071388944
Epoch 7, Training Loss: 0.2673, Testing Loss: 0.2962767115444135
Epoch 8, Training Loss: 0.2639, Testing Loss: 0.30444807719923994
Epoch 9, Training Loss: 0.2612, Testing Loss: 0.29512267998395125
Epoch 10, Training Loss: 0.2563, Testing Loss: 0.2937116707531944
Epoch 11, Training Loss: 0.2547, Testing Loss: 0.29158888216231277
Epoch 12, Training Loss: 0.2513, Testing Loss: 0.29475830697384175
Epoch 13, Training Loss: 0.2490, Testing Loss: 0.29085357080436597
Epoch 14, Training Loss: 0.2461, Testing Loss: 0.29103921322631665
Epoch 15, Training Loss: 0.2437, Testing Loss: 0.2882105998552529
Epoch 16,

In [123]:
all_test_preds, all_test_labels = run_inference(model, test_loader)

In [124]:
# Per-class accuracy
def calc_per_class_accs(all_test_preds, all_test_labels):
    per_class_acc = (all_test_preds == all_test_labels).float().mean(dim=0)  # [num_classes]
    # Print results
    for idx, acc in enumerate(per_class_acc):
        print(f"Class {mlb.classes_[idx]} accuracy: {acc.item():.3f}")
    print('\n')

In [125]:
from sklearn.metrics import f1_score
def calc_f1_scores(all_test_labels, all_test_preds, average=None):
    f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
    for idx, score in enumerate(f1_scores):
        print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")
    print('\n')

In [126]:
from sklearn.metrics import roc_auc_score

def calc_roc_auc_scores(all_test_labels, all_test_preds):
    num_classes = all_test_labels.shape[1]
    aucs = []

    for i in range(num_classes):
        try:
            auc = roc_auc_score(all_test_labels[:, i], all_test_preds[:, i])
        except ValueError:
            # happens if class i has no positive samples in test set
            auc = float('nan')
        aucs.append(auc)

    for idx, auc in enumerate(aucs):
        print(f"Class {mlb.classes_[idx]} AUROC: {auc:.3f}")
    print('\n')

In [127]:
print('-----CNN Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----CNN Test Evals-----
Class CD accuracy: 0.890
Class HYP accuracy: 0.916
Class MI accuracy: 0.872
Class NORM accuracy: 0.864
Class STTC accuracy: 0.889


Class CD F1-score: 0.725
Class HYP F1-score: 0.576
Class MI F1-score: 0.712
Class NORM F1-score: 0.853
Class STTC F1-score: 0.768


Class CD AUROC: 0.802
Class HYP AUROC: 0.726
Class MI AUROC: 0.793
Class NORM AUROC: 0.868
Class STTC AUROC: 0.850




### --- GRU Model ---

In [149]:
# GRU wants batched input as (B, L, H_in), unlike for our CNN model
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [150]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [151]:
import torch.nn as nn
import torch.nn.functional as F

# following kaggle tutorial here https://www.kaggle.com/code/fanbyprinciple/learning-pytorch-3-coding-an-rnn-gru-lstm
class ECGGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length, bidirectional):
        super(ECGGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bias=True,
            batch_first=True,
            bidirectional=bidirectional,
        )
        self.fc1 = nn.Linear(hidden_size*sequence_length*(1+bidirectional), num_classes)  # if bidirectional, the output size is doubled, so (1+bidrectional) accounts for that

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        out, _ = self.gru(x, h0)
        out = out.reshape(out.shape[0], -1)
        out = self.fc1(out)
        return out

In [144]:
model = ECGGRU(
    input_size=12,
    hidden_size=32,
    num_layers=1,
    num_classes=5,
    sequence_length=1000,
    bidirectional=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, 'gru_best_model.pt'), num_epochs=30)

Epoch 1, Training Loss: 0.5190, Testing Loss: 0.4743
Epoch 2, Training Loss: 0.4396, Testing Loss: 0.4421
Epoch 3, Training Loss: 0.3862, Testing Loss: 0.4310
Epoch 4, Training Loss: 0.3511, Testing Loss: 0.4240
Epoch 5, Training Loss: 0.3240, Testing Loss: 0.4306
Epoch 6, Training Loss: 0.3020, Testing Loss: 0.4296
Epoch 7, Training Loss: 0.2809, Testing Loss: 0.4394
Epoch 8, Training Loss: 0.2602, Testing Loss: 0.4524
Epoch 9, Training Loss: 0.2421, Testing Loss: 0.4760
Epoch 10, Training Loss: 0.2245, Testing Loss: 0.4845
Epoch 11, Training Loss: 0.2052, Testing Loss: 0.5058
Epoch 12, Training Loss: 0.1884, Testing Loss: 0.5169
Epoch 13, Training Loss: 0.1714, Testing Loss: 0.5417
Epoch 14, Training Loss: 0.1570, Testing Loss: 0.5717
Epoch 15, Training Loss: 0.1417, Testing Loss: 0.6151
Epoch 16, Training Loss: 0.1270, Testing Loss: 0.6242
Epoch 17, Training Loss: 0.1161, Testing Loss: 0.6601
Epoch 18, Training Loss: 0.1037, Testing Loss: 0.6807
Epoch 19, Training Loss: 0.0914, Test

In [152]:
def run_inference_gru(model, dataloader):
    model.eval()
    all_test_preds, all_test_labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_test_preds.append(preds.cpu())
            all_test_labels.append(y.cpu())

    # Concatenate all batches
    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    return all_test_preds, all_test_labels

In [153]:
# load the best model from GRU training
best_gru_model = ECGGRU(
    input_size=12,
    hidden_size=32,
    num_layers=1,
    num_classes=5,
    sequence_length=1000,
    bidirectional=False
).to(device)

checkpoint_path = "/content/drive/MyDrive/gru_best_model.pt"
best_gru_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [154]:
all_test_preds, all_test_labels = run_inference(best_gru_model, test_loader)

In [155]:
print('-----GRU Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----GRU Test Evals-----
Class CD accuracy: 0.832
Class HYP accuracy: 0.895
Class MI accuracy: 0.793
Class NORM accuracy: 0.783
Class STTC accuracy: 0.825


Class CD F1-score: 0.533
Class HYP F1-score: 0.346
Class MI F1-score: 0.486
Class NORM F1-score: 0.776
Class STTC F1-score: 0.518


Class CD AUROC: 0.688
Class HYP AUROC: 0.609
Class MI AUROC: 0.659
Class NORM AUROC: 0.791
Class STTC AUROC: 0.677


